# Verify Data Quality

This notebook performs basic data quality checks on the Gab dataset, including:
- Loading posts_current_snapshot and posts_incremental files
- Checking for missing values
- Identifying columns with all null values
- Basic statistics about data completeness

## Import Libraries

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## Define Data Paths

Let's identify all available snapshot folders in the data/raw directory.

In [4]:
# Define base path
data_path = Path('../data/raw')

# List all snapshot folders
snapshot_folders = [f for f in data_path.iterdir() if f.is_dir()]
print(f"Available snapshots: {len(snapshot_folders)}")
for folder in sorted(snapshot_folders):
    print(f"  - {folder.name}")

Available snapshots: 6
  - 2016-2021
  - 2022
  - 2023
  - 2024
  - Jan-Jul 25
  - Jul 25


## Load Data from All Snapshots

Load both current snapshot and incremental data from all available snapshots.

In [5]:
# Initialize dictionaries to store all snapshots
snapshots_data = {}  # {snapshot_name: {'current': df, 'incremental': df}}
df_current_all = None  # Aggregated current snapshots
df_incremental_all = None  # Aggregated incremental data

print("Loading data from all snapshots...")
print(f"Total snapshots found: {len(snapshot_folders)}\n")

for snapshot in sorted(snapshot_folders):
    snapshot_name = snapshot.name
    current_path = snapshot / 'posts_current_snapshot.csv'
    incremental_path = snapshot / 'posts_incremental.csv'
    
    snapshots_data[snapshot_name] = {'current': None, 'incremental': None}
    
    # Load current snapshot
    if current_path.exists():
        try:
            df = pd.read_csv(current_path)
            snapshots_data[snapshot_name]['current'] = df
            print(f"\u2713 {snapshot_name}: posts_current_snapshot.csv loaded ({df.shape[0]:,} rows)")
        except Exception as e:
            print(f"\u2717 {snapshot_name}: Error loading posts_current_snapshot.csv - {e}")
    else:
        print(f"\u2717 {snapshot_name}: posts_current_snapshot.csv not found")
    
    # Load incremental data
    if incremental_path.exists():
        try:
            df = pd.read_csv(incremental_path)
            snapshots_data[snapshot_name]['incremental'] = df
            print(f"\u2713 {snapshot_name}: posts_incremental.csv loaded ({df.shape[0]:,} rows)")
        except Exception as e:
            print(f"\u2717 {snapshot_name}: Error loading posts_incremental.csv - {e}")
    else:
        print(f"\u2717 {snapshot_name}: posts_incremental.csv not found")

# Create aggregated datasets
print(f"\nCreating aggregated datasets...")
current_dfs = [data['current'] for data in snapshots_data.values() if data['current'] is not None]
incremental_dfs = [data['incremental'] for data in snapshots_data.values() if data['incremental'] is not None]

if current_dfs:
    df_current_all = pd.concat(current_dfs, ignore_index=True)
    print(f"\u2713 Aggregated current snapshots: {df_current_all.shape[0]:,} rows")
else:
    print(f"\u2717 No current snapshot data available")

if incremental_dfs:
    df_incremental_all = pd.concat(incremental_dfs, ignore_index=True)
    print(f"\u2713 Aggregated incremental data: {df_incremental_all.shape[0]:,} rows")
else:
    print(f"\u2717 No incremental data available")

# Set df_current for backward compatibility with existing code
if snapshot_folders:
    first_snapshot = sorted(snapshot_folders)[0]
    df_current = snapshots_data[first_snapshot.name]['current']
    df_incremental = snapshots_data[first_snapshot.name]['incremental']
else:
    df_current = None
    df_incremental = None

Loading data from all snapshots...
Total snapshots found: 6

✓ 2016-2021: posts_current_snapshot.csv loaded (18,926 rows)
✓ 2016-2021: posts_incremental.csv loaded (18,926 rows)
✓ 2022: posts_current_snapshot.csv loaded (13,108 rows)
✓ 2022: posts_incremental.csv loaded (32,034 rows)
✓ 2023: posts_current_snapshot.csv loaded (13,376 rows)
✓ 2023: posts_incremental.csv loaded (45,410 rows)
✓ 2024: posts_current_snapshot.csv loaded (13,400 rows)
✓ 2024: posts_incremental.csv loaded (58,810 rows)
✓ Jan-Jul 25: posts_current_snapshot.csv loaded (44,878 rows)
✓ Jan-Jul 25: posts_incremental.csv loaded (103,688 rows)
✓ Jul 25: posts_current_snapshot.csv loaded (78,857 rows)
✓ Jul 25: posts_incremental.csv loaded (182,545 rows)

Creating aggregated datasets...
✓ Aggregated current snapshots: 182,545 rows
✓ Aggregated incremental data: 441,413 rows


## Data Quality Check 1: Missing Values Overview

Check the percentage of missing values for each column in all snapshots and aggregates.

In [6]:
def check_missing_values(df, dataset_name):
    """Check missing values in the dataframe"""
    print(f"\n{'='*60}")
    print(f"Missing Values Analysis - {dataset_name}")
    print(f"{'='*60}\n")
    
    # Calculate missing values
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    
    # Create summary dataframe
    missing_df = pd.DataFrame({
        'Column': df.columns,
        'Missing_Count': missing.values,
        'Missing_Percentage': missing_pct.values
    })
    
    # Sort by missing percentage
    missing_df = missing_df.sort_values('Missing_Percentage', ascending=False)
    
    print(missing_df.to_string(index=False))
    
    # Identify columns with all null values
    all_null_cols = missing_df[missing_df['Missing_Percentage'] == 100.0]['Column'].tolist()
    if all_null_cols:
        print(f"\n⚠️  Columns with ALL null values: {all_null_cols}")
    else:
        print(f"\n✓ No columns with all null values")
    
    # Identify columns with some null values
    some_null_cols = missing_df[(missing_df['Missing_Percentage'] > 0) & 
                                 (missing_df['Missing_Percentage'] < 100.0)]['Column'].tolist()
    if some_null_cols:
        print(f"⚠️  Columns with SOME null values: {len(some_null_cols)}")
    
    return missing_df

# Check each snapshot
for snapshot_name, data in sorted(snapshots_data.items()):
    if data['current'] is not None:
        missing_current = check_missing_values(data['current'], f"posts_current_snapshot - {snapshot_name}")
    if data['incremental'] is not None:
        missing_incremental = check_missing_values(data['incremental'], f"posts_incremental - {snapshot_name}")


Missing Values Analysis - posts_current_snapshot - 2016-2021

          Column  Missing_Count  Missing_Percentage
        language            325            1.717214
              id              0            0.000000
      Unnamed: 0              0            0.000000
         content              0            0.000000
      account_id              0            0.000000
      created_at              0            0.000000
   replies_count              0            0.000000
   reblogs_count              0            0.000000
reactions_counts              0            0.000000
    quotes_count              0            0.000000
       timestamp              0            0.000000

✓ No columns with all null values
⚠️  Columns with SOME null values: 1

Missing Values Analysis - posts_incremental - 2016-2021

          Column  Missing_Count  Missing_Percentage
        language            325            1.717214
              id              0            0.000000
      Unnamed: 0           

In [7]:
# Check aggregated data
if df_current_all is not None:
    print("\n" + "#"*80)
    print("AGGREGATE ANALYSIS - ALL SNAPSHOTS COMBINED")
    print("#"*80)
    missing_current_all = check_missing_values(df_current_all, "posts_current_snapshot - ALL AGGREGATED")

if df_incremental_all is not None:
    missing_incremental_all = check_missing_values(df_incremental_all, "posts_incremental - ALL AGGREGATED")


################################################################################
AGGREGATE ANALYSIS - ALL SNAPSHOTS COMBINED
################################################################################

Missing Values Analysis - posts_current_snapshot - ALL AGGREGATED

          Column  Missing_Count  Missing_Percentage
        language            877            0.480429
              id              0            0.000000
      Unnamed: 0              0            0.000000
         content              0            0.000000
      account_id              0            0.000000
      created_at              0            0.000000
   replies_count              0            0.000000
   reblogs_count              0            0.000000
reactions_counts              0            0.000000
    quotes_count              0            0.000000
       timestamp              0            0.000000

✓ No columns with all null values
⚠️  Columns with SOME null values: 1

Missing Values Analysis - po

## Data Quality Check 2: Rows with Missing Values

Check how many rows have at least one missing value in all snapshots and aggregates.

In [8]:
def check_rows_with_missing(df, dataset_name):
    """Check rows with missing values"""
    print(f"\n{'='*60}")
    print(f"Rows with Missing Values - {dataset_name}")
    print(f"{'='*60}\n")
    
    total_rows = len(df)
    rows_with_missing = df.isnull().any(axis=1).sum()
    rows_with_missing_pct = (rows_with_missing / total_rows) * 100
    
    print(f"Total rows: {total_rows:,}")
    print(f"Rows with at least one missing value: {rows_with_missing:,} ({rows_with_missing_pct:.2f}%)")
    print(f"Complete rows (no missing values): {total_rows - rows_with_missing:,} ({100 - rows_with_missing_pct:.2f}%)")
    
    # show the first few rows with missing values (limit to 5 to avoid clutter)
    if rows_with_missing > 0:
        print(f"\nFirst few rows with missing values:")
        display(df[df.isnull().any(axis=1)].head(5)) # ty: ignore[unresolved-reference]

    return rows_with_missing, rows_with_missing_pct

# Check each snapshot
for snapshot_name, data in sorted(snapshots_data.items()):
    if data['current'] is not None:
        check_rows_with_missing(data['current'], f"posts_current_snapshot - {snapshot_name}")
    if data['incremental'] is not None:
        check_rows_with_missing(data['incremental'], f"posts_incremental - {snapshot_name}")


Rows with Missing Values - posts_current_snapshot - 2016-2021

Total rows: 18,926
Rows with at least one missing value: 325 (1.72%)
Complete rows (no missing values): 18,601 (98.28%)

First few rows with missing values:


,Unnamed: 0,id,content,account_id,created_at,language,replies_count,reblogs_count,reactions_counts,quotes_count,timestamp
875,875,106058693145876424,picturesilove,107776,2021-04-13T15:20:26.968Z,NaN,0,1,{'1': 9},0,2021-04-13 15:20:26.968000+00:00
876,876,106053318471727374,BlackandWhiteMondays,107776,2021-04-12T16:33:35.941Z,NaN,1,0,{'1': 5},0,2021-04-12 16:33:35.941000+00:00
877,877,106036076656799434,picturesilove,107776,2021-04-09T15:28:46.729Z,NaN,1,3,{'1': 8},0,2021-04-09 15:28:46.729000+00:00
878,878,106036051862555871,FreakyFriday,107776,2021-04-09T15:22:28.390Z,NaN,1,2,{'1': 7},0,2021-04-09 15:22:28.390000+00:00
880,880,106002211401465441,picturesilove,107776,2021-04-03T15:56:23.934Z,NaN,0,1,{'1': 6},0,2021-04-03 15:56:23.934000+00:00



Rows with Missing Values - posts_incremental - 2016-2021

Total rows: 18,926
Rows with at least one missing value: 325 (1.72%)
Complete rows (no missing values): 18,601 (98.28%)

First few rows with missing values:


,Unnamed: 0,id,content,account_id,created_at,language,replies_count,reblogs_count,reactions_counts,quotes_count,timestamp
875,875,106058693145876424,picturesilove,107776,2021-04-13T15:20:26.968Z,NaN,0,1,{'1': 9},0,2021-04-13 15:20:26.968000+00:00
876,876,106053318471727374,BlackandWhiteMondays,107776,2021-04-12T16:33:35.941Z,NaN,1,0,{'1': 5},0,2021-04-12 16:33:35.941000+00:00
877,877,106036076656799434,picturesilove,107776,2021-04-09T15:28:46.729Z,NaN,1,3,{'1': 8},0,2021-04-09 15:28:46.729000+00:00
878,878,106036051862555871,FreakyFriday,107776,2021-04-09T15:22:28.390Z,NaN,1,2,{'1': 7},0,2021-04-09 15:22:28.390000+00:00
880,880,106002211401465441,picturesilove,107776,2021-04-03T15:56:23.934Z,NaN,0,1,{'1': 6},0,2021-04-03 15:56:23.934000+00:00



Rows with Missing Values - posts_current_snapshot - 2022

Total rows: 13,108
Rows with at least one missing value: 256 (1.95%)
Complete rows (no missing values): 12,852 (98.05%)

First few rows with missing values:


...

,Unnamed: 0,id,content,account_id,created_at,language,replies_count,reblogs_count,reactions_counts,quotes_count,timestamp
875,875,106058693145876424,picturesilove,107776,2021-04-13T15:20:26.968Z,NaN,0,1,{'1': 9},0,2021-04-13 15:20:26.968000+00:00
876,876,106053318471727374,BlackandWhiteMondays,107776,2021-04-12T16:33:35.941Z,NaN,1,0,{'1': 5},0,2021-04-12 16:33:35.941000+00:00
877,877,106036076656799434,picturesilove,107776,2021-04-09T15:28:46.729Z,NaN,1,3,{'1': 8},0,2021-04-09 15:28:46.729000+00:00
878,878,106036051862555871,FreakyFriday,107776,2021-04-09T15:22:28.390Z,NaN,1,2,{'1': 7},0,2021-04-09 15:22:28.390000+00:00
880,880,106002211401465441,picturesilove,107776,2021-04-03T15:56:23.934Z,NaN,0,1,{'1': 6},0,2021-04-03 15:56:23.934000+00:00



Rows with Missing Values - posts_current_snapshot - Jul 25

Total rows: 78,857
Rows with at least one missing value: 179 (0.23%)
Complete rows (no missing values): 78,678 (99.77%)

First few rows with missing values:


,Unnamed: 0,id,content,account_id,created_at,language,replies_count,reblogs_count,reactions_counts,quotes_count,timestamp
29,29,114936368019849175,TX Invasion Pedophilia,418729,2025-07-29T11:49:55.229Z,NaN,2,7,{'1': 7},0,2025-07-29 11:49:55.229000+00:00
1268,1268,114931383088035247,GabFamMartaVonRunge blat1982 Feralfae Joycelyn...,29492,2025-07-28T14:42:11.228Z,NaN,0,1,"{'6': 1, '1': 3}",0,2025-07-28 14:42:11.228000+00:00
1270,1270,114920026491174891,GabFamMartaVonRunge blat1982 Feralfae Joycelyn...,29492,2025-07-26T14:34:03.290Z,NaN,1,1,"{'6': 1, '1': 3}",0,2025-07-26 14:34:03.290000+00:00
1271,1271,114914360664621180,GabFamMartaVonRunge blat1982 Feralfae Joycelyn...,29492,2025-07-25T14:33:09.644Z,NaN,0,2,"{'6': 1, '1': 8}",0,2025-07-25 14:33:09.644000+00:00
1272,1272,114908726230470386,GabFamMartaVonRunge blat1982 Feralfae Joycelyn...,29492,2025-07-24T14:40:14.998Z,NaN,1,4,"{'6': 3, '1': 7}",0,2025-07-24 14:40:14.998000+00:00



Rows with Missing Values - posts_incremental - Jul 25

Total rows: 182,545
Rows with at least one missing value: 877 (0.48%)
Complete rows (no missing values): 181,668 (99.52%)

First few rows with missing values:


,Unnamed: 0,id,content,account_id,created_at,language,replies_count,reblogs_count,reactions_counts,quotes_count,timestamp
875,875,106058693145876424,picturesilove,107776,2021-04-13T15:20:26.968Z,NaN,0,1,{'1': 9},0,2021-04-13 15:20:26.968000+00:00
876,876,106053318471727374,BlackandWhiteMondays,107776,2021-04-12T16:33:35.941Z,NaN,1,0,{'1': 5},0,2021-04-12 16:33:35.941000+00:00
877,877,106036076656799434,picturesilove,107776,2021-04-09T15:28:46.729Z,NaN,1,3,{'1': 8},0,2021-04-09 15:28:46.729000+00:00
878,878,106036051862555871,FreakyFriday,107776,2021-04-09T15:22:28.390Z,NaN,1,2,{'1': 7},0,2021-04-09 15:22:28.390000+00:00
880,880,106002211401465441,picturesilove,107776,2021-04-03T15:56:23.934Z,NaN,0,1,{'1': 6},0,2021-04-03 15:56:23.934000+00:00


In [18]:
# Check aggregated data
if df_current_all is not None:
    check_rows_with_missing(df_current_all, "posts_current_snapshot - ALL AGGREGATED")

if df_incremental_all is not None:
    check_rows_with_missing(df_incremental_all, "posts_incremental - ALL AGGREGATED")


Rows with Missing Values - posts_current_snapshot - ALL AGGREGATED

Total rows: 182,545
Rows with at least one missing value: 877 (0.48%)
Complete rows (no missing values): 181,668 (99.52%)

First few rows with missing values:


,Unnamed: 0,id,content,account_id,created_at,language,replies_count,reblogs_count,reactions_counts,quotes_count,timestamp
875,875,106058693145876424,picturesilove,107776,2021-04-13T15:20:26.968Z,NaN,0,1,{'1': 9},0,2021-04-13 15:20:26.968000+00:00
876,876,106053318471727374,BlackandWhiteMondays,107776,2021-04-12T16:33:35.941Z,NaN,1,0,{'1': 5},0,2021-04-12 16:33:35.941000+00:00
877,877,106036076656799434,picturesilove,107776,2021-04-09T15:28:46.729Z,NaN,1,3,{'1': 8},0,2021-04-09 15:28:46.729000+00:00
878,878,106036051862555871,FreakyFriday,107776,2021-04-09T15:22:28.390Z,NaN,1,2,{'1': 7},0,2021-04-09 15:22:28.390000+00:00
880,880,106002211401465441,picturesilove,107776,2021-04-03T15:56:23.934Z,NaN,0,1,{'1': 6},0,2021-04-03 15:56:23.934000+00:00



Rows with Missing Values - posts_incremental - ALL AGGREGATED

Total rows: 441,413
Rows with at least one missing value: 3,747 (0.85%)
Complete rows (no missing values): 437,666 (99.15%)

First few rows with missing values:


,Unnamed: 0,id,content,account_id,created_at,language,replies_count,reblogs_count,reactions_counts,quotes_count,timestamp
875,875,106058693145876424,picturesilove,107776,2021-04-13T15:20:26.968Z,NaN,0,1,{'1': 9},0,2021-04-13 15:20:26.968000+00:00
876,876,106053318471727374,BlackandWhiteMondays,107776,2021-04-12T16:33:35.941Z,NaN,1,0,{'1': 5},0,2021-04-12 16:33:35.941000+00:00
877,877,106036076656799434,picturesilove,107776,2021-04-09T15:28:46.729Z,NaN,1,3,{'1': 8},0,2021-04-09 15:28:46.729000+00:00
878,878,106036051862555871,FreakyFriday,107776,2021-04-09T15:22:28.390Z,NaN,1,2,{'1': 7},0,2021-04-09 15:22:28.390000+00:00
880,880,106002211401465441,picturesilove,107776,2021-04-03T15:56:23.934Z,NaN,0,1,{'1': 6},0,2021-04-03 15:56:23.934000+00:00


## Data Quality Check 3: Data Types

Verify the data types of each column match expectations across all snapshots and aggregates.

In [10]:
def check_data_types(df, dataset_name):
    """Check data types of columns"""
    print(f"\n{'='*60}")
    print(f"Data Types - {dataset_name}")
    print(f"{'='*60}\n")
    
    dtypes_df = pd.DataFrame({
        'Column': df.columns,
        'Data_Type': df.dtypes.values,
        'Non_Null_Count': df.count().values,
        'Null_Count': df.isnull().sum().values
    })
    
    print(dtypes_df.to_string(index=False))
    
    return dtypes_df

# Check each snapshot
for snapshot_name, data in sorted(snapshots_data.items()):
    if data['current'] is not None:
        dtypes_current = check_data_types(data['current'], f"posts_current_snapshot - {snapshot_name}")
    if data['incremental'] is not None:
        dtypes_incremental = check_data_types(data['incremental'], f"posts_incremental - {snapshot_name}")


Data Types - posts_current_snapshot - 2016-2021

          Column Data_Type  Non_Null_Count  Null_Count
      Unnamed: 0     int64           18926           0
              id     int64           18926           0
         content       str           18926           0
      account_id     int64           18926           0
      created_at       str           18926           0
        language       str           18601         325
   replies_count     int64           18926           0
   reblogs_count     int64           18926           0
reactions_counts       str           18926           0
    quotes_count     int64           18926           0
       timestamp       str           18926           0

Data Types - posts_incremental - 2016-2021

          Column Data_Type  Non_Null_Count  Null_Count
      Unnamed: 0     int64           18926           0
              id     int64           18926           0
         content       str           18926           0
      account_id     int6

In [11]:
# Check aggregated data
if df_current_all is not None:
    dtypes_current_all = check_data_types(df_current_all, "posts_current_snapshot - ALL AGGREGATED")

if df_incremental_all is not None:
    dtypes_incremental_all = check_data_types(df_incremental_all, "posts_incremental - ALL AGGREGATED")


Data Types - posts_current_snapshot - ALL AGGREGATED

          Column Data_Type  Non_Null_Count  Null_Count
      Unnamed: 0     int64          182545           0
              id     int64          182545           0
         content       str          182545           0
      account_id     int64          182545           0
      created_at       str          182545           0
        language       str          181668         877
   replies_count     int64          182545           0
   reblogs_count     int64          182545           0
reactions_counts       str          182545           0
    quotes_count     int64          182545           0
       timestamp       str          182545           0

Data Types - posts_incremental - ALL AGGREGATED

          Column Data_Type  Non_Null_Count  Null_Count
      Unnamed: 0     int64          441413           0
              id     int64          441413           0
         content       str          441413           0
      account_i

## Data Quality Check 4: Basic Statistics

Get basic statistics for numerical columns in all snapshots and aggregates.

In [12]:
# Statistics for each snapshot
for snapshot_name, data in sorted(snapshots_data.items()):
    if data['current'] is not None:
        print(f"\nBasic Statistics - posts_current_snapshot ({snapshot_name})")
        print("="*60)
        display(data['current'].describe()) # ty: ignore[unresolved-reference]


Basic Statistics - posts_current_snapshot (2016-2021)


,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,18926.000000,1.892600e+04,1.892600e+04,18926.000000,18926.000000,18926.000000
mean,9462.500000,1.040447e+17,2.138723e+06,33.311371,118.277608,4.409225
std,5463.609933,1.367338e+16,1.603552e+06,221.709910,548.793006,23.724165
min,0.000000,2.026391e+15,1.000000e+00,0.000000,0.000000,0.000000
25%,4731.250000,1.055977e+17,5.918070e+05,1.000000,2.000000,0.000000
50%,9462.500000,1.060147e+17,1.977327e+06,4.000000,10.000000,0.000000
75%,14193.750000,1.067716e+17,3.424491e+06,15.000000,49.000000,2.000000
max,18925.000000,1.075441e+17,5.628397e+06,14899.000000,24464.000000,1275.000000



Basic Statistics - posts_current_snapshot (2022)


,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,13108.000000,1.310800e+04,1.310800e+04,13108.000000,13108.000000,13108.000000
mean,6553.500000,1.085484e+17,3.066766e+06,21.990616,89.721849,4.077662
std,3784.097999,6.271603e+14,2.054556e+06,206.620688,297.507483,16.035307
min,0.000000,1.075445e+17,3.100000e+01,0.000000,0.000000,0.000000
25%,3276.750000,1.079529e+17,1.246183e+06,0.000000,1.000000,0.000000
50%,6553.500000,1.085407e+17,2.957838e+06,2.000000,6.000000,0.000000
75%,9830.250000,1.091188e+17,5.210017e+06,10.000000,32.250000,2.000000
max,13107.000000,1.096110e+17,6.195143e+06,22211.000000,6246.000000,514.000000



Basic Statistics - posts_current_snapshot (2023)


...

,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,13400.000000,1.340000e+04,1.340000e+04,13400.000000,13400.000000,13400.000000
mean,6699.500000,1.126885e+17,3.263658e+06,9.828284,28.618657,2.636940
std,3868.391138,6.265002e+14,2.299302e+06,45.233010,84.093757,28.611694
min,0.000000,1.116784e+17,3.100000e+01,0.000000,0.000000,0.000000
25%,3349.750000,1.120629e+17,1.105452e+06,0.000000,0.000000,0.000000
50%,6699.500000,1.127099e+17,3.238878e+06,1.000000,3.000000,0.000000
75%,10049.250000,1.132525e+17,5.552423e+06,4.000000,12.000000,1.000000
max,13399.000000,1.137501e+17,6.679919e+06,2432.000000,1802.000000,2292.000000



Basic Statistics - posts_current_snapshot (Jan-Jul 25)


,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,44878.000000,4.487800e+04,4.487800e+04,44878.000000,44878.000000,44878.000000
mean,22440.927403,1.144971e+17,2.887835e+06,3.879184,10.903962,0.825549
std,12958.353353,2.266520e+14,2.211556e+06,30.204602,39.419591,3.448387
min,0.000000,1.137502e+17,2.700000e+01,0.000000,0.000000,0.000000
25%,11219.250000,1.144324e+17,8.698450e+05,0.000000,0.000000,0.000000
50%,22438.500000,1.146157e+17,2.435241e+06,0.000000,1.000000,0.000000
75%,33663.750000,1.146273e+17,4.955828e+06,2.000000,4.000000,0.000000
max,44888.000000,1.147750e+17,6.729793e+06,5170.000000,766.000000,190.000000



Basic Statistics - posts_current_snapshot (Jul 25)


,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,78857.000000,7.885700e+04,7.885700e+04,78857.000000,78857.000000,78857.000000
mean,39428.000000,1.149259e+17,2.805005e+06,2.106192,5.227247,0.442535
std,22764.199426,2.553616e+13,2.153364e+06,8.201147,17.909935,1.774279
min,0.000000,1.147751e+17,2.700000e+01,0.000000,0.000000,0.000000
25%,19714.000000,1.149193e+17,7.724140e+05,0.000000,0.000000,0.000000
50%,39428.000000,1.149323e+17,2.370516e+06,0.000000,1.000000,0.000000
75%,59142.000000,1.149391e+17,4.578740e+06,1.000000,3.000000,0.000000
max,78856.000000,1.149606e+17,6.734613e+06,481.000000,500.000000,165.000000


In [13]:
# Statistics for incremental data in each snapshot
for snapshot_name, data in sorted(snapshots_data.items()):
    if data['incremental'] is not None:
        print(f"\nBasic Statistics - posts_incremental ({snapshot_name})")
        print("="*60)
        display(data['incremental'].describe()) # ty: ignore[unresolved-reference]


Basic Statistics - posts_incremental (2016-2021)


,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,18926.000000,1.892600e+04,1.892600e+04,18926.000000,18926.000000,18926.000000
mean,9462.500000,1.040447e+17,2.138723e+06,33.311371,118.277608,4.409225
std,5463.609933,1.367338e+16,1.603552e+06,221.709910,548.793006,23.724165
min,0.000000,2.026391e+15,1.000000e+00,0.000000,0.000000,0.000000
25%,4731.250000,1.055977e+17,5.918070e+05,1.000000,2.000000,0.000000
50%,9462.500000,1.060147e+17,1.977327e+06,4.000000,10.000000,0.000000
75%,14193.750000,1.067716e+17,3.424491e+06,15.000000,49.000000,2.000000
max,18925.000000,1.075441e+17,5.628397e+06,14899.000000,24464.000000,1275.000000



Basic Statistics - posts_incremental (2022)


,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,32034.000000,3.203400e+04,3.203400e+04,32034.000000,32034.000000,32034.000000
mean,16016.909190,1.058876e+17,2.518469e+06,28.679029,106.592870,4.273553
std,9247.982338,1.074806e+16,1.858650e+06,215.731742,462.974918,20.922695
min,0.000000,2.026391e+15,1.000000e+00,0.000000,0.000000,0.000000
25%,8008.250000,1.058315e+17,8.549190e+05,0.000000,1.000000,0.000000
50%,16016.500000,1.070410e+17,2.206090e+06,3.000000,8.000000,0.000000
75%,24025.750000,1.082639e+17,3.973898e+06,13.000000,43.000000,2.000000
max,32034.000000,1.096110e+17,6.195143e+06,22211.000000,24464.000000,1275.000000



Basic Statistics - posts_incremental (2023)


...

,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,58810.000000,5.881000e+04,5.881000e+04,58810.000000,58810.000000,58810.000000
mean,29406.317038,1.085175e+17,2.872677e+06,20.518721,74.512090,3.613535
std,16978.850795,8.476514e+15,2.091392e+06,162.124401,352.242081,21.389257
min,0.000000,2.026391e+15,1.000000e+00,0.000000,0.000000,0.000000
25%,14702.250000,1.068405e+17,1.008774e+06,0.000000,1.000000,0.000000
50%,29405.500000,1.092255e+17,2.474260e+06,2.000000,5.000000,0.000000
75%,44108.750000,1.114797e+17,4.842301e+06,9.000000,27.000000,2.000000
max,58814.000000,1.137501e+17,6.679919e+06,22211.000000,24464.000000,2292.000000



Basic Statistics - posts_incremental (Jan-Jul 25)


,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,103688.000000,1.036880e+05,1.036880e+05,103688.000000,103688.000000,103688.000000
mean,51883.236353,1.111056e+17,2.879238e+06,13.316835,46.981367,2.406846
std,29970.674895,7.039353e+15,2.144231e+06,123.978726,268.399200,16.326024
min,0.000000,2.026391e+15,1.000000e+00,0.000000,0.000000,0.000000
25%,25922.750000,1.086238e+17,9.466000e+05,0.000000,0.000000,0.000000
50%,51848.500000,1.126480e+17,2.464646e+06,1.000000,2.000000,0.000000
75%,77852.250000,1.146059e+17,4.864228e+06,5.000000,13.000000,1.000000
max,103785.000000,1.147750e+17,6.729793e+06,22211.000000,24464.000000,2292.000000



Basic Statistics - posts_incremental (Jul 25)


,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,182545.000000,1.825450e+05,1.825450e+05,182545.000000,182545.000000,182545.000000
mean,93457.959840,1.127559e+17,2.847170e+06,8.473987,28.944145,1.558290
std,54834.355796,5.632742e+15,2.148490e+06,93.758379,203.678191,12.397734
min,0.000000,2.026391e+15,1.000000e+00,0.000000,0.000000,0.000000
25%,45641.000000,1.117115e+17,8.676300e+05,0.000000,0.000000,0.000000
50%,91363.000000,1.146269e+17,2.422857e+06,0.000000,1.000000,0.000000
75%,141916.000000,1.149287e+17,4.825861e+06,3.000000,7.000000,1.000000
max,187552.000000,1.149606e+17,6.734613e+06,22211.000000,24464.000000,2292.000000


In [14]:
# Statistics for aggregated data
if df_current_all is not None:
    print("\n" + "#"*80)
    print("AGGREGATE ANALYSIS - ALL SNAPSHOTS COMBINED")
    print("#"*80)
    print("\nBasic Statistics - posts_current_snapshot (ALL AGGREGATED)")
    print("="*60)
    display(df_current_all.describe()) # ty: ignore[unresolved-reference]


################################################################################
AGGREGATE ANALYSIS - ALL SNAPSHOTS COMBINED
################################################################################

Basic Statistics - posts_current_snapshot (ALL AGGREGATED)


,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,182545.000000,1.825450e+05,1.825450e+05,182545.000000,182545.000000,182545.000000
mean,24982.845813,1.127559e+17,2.847170e+06,8.473987,28.944145,1.558290
std,21485.105957,5.632742e+15,2.148490e+06,93.758379,203.678191,12.397734
min,0.000000,2.026391e+15,1.000000e+00,0.000000,0.000000,0.000000
25%,7606.000000,1.117115e+17,8.676300e+05,0.000000,0.000000,0.000000
50%,17129.000000,1.146269e+17,2.422857e+06,0.000000,1.000000,0.000000
75%,39053.000000,1.149287e+17,4.825861e+06,3.000000,7.000000,1.000000
max,78856.000000,1.149606e+17,6.734613e+06,22211.000000,24464.000000,2292.000000


In [15]:
if df_incremental_all is not None:
    print("\nBasic Statistics - posts_incremental (ALL AGGREGATED)")
    print("="*60)
    display(df_incremental_all.describe()) # ty: ignore[unresolved-reference]


Basic Statistics - posts_incremental (ALL AGGREGATED)


,Unnamed: 0,id,account_id,replies_count,reblogs_count,quotes_count
count,441413.000000,4.414130e+05,4.414130e+05,441413.000000,441413.000000,441413.000000
mean,58658.358372,1.103690e+17,2.794626e+06,15.311167,54.798461,2.591804
std,50123.074542,8.166958e+15,2.092034e+06,140.448457,310.051520,16.739242
min,0.000000,2.026391e+15,1.000000e+00,0.000000,0.000000,0.000000
25%,18392.000000,1.074470e+17,9.085520e+05,0.000000,0.000000,0.000000
50%,42438.000000,1.112311e+17,2.385588e+06,1.000000,3.000000,0.000000
75%,88028.000000,1.146252e+17,4.561833e+06,5.000000,15.000000,1.000000
max,187552.000000,1.149606e+17,6.734613e+06,22211.000000,24464.000000,2292.000000


## Data Quality Check 5: Snapshot vs Incremental Consistency

Check if all posts in the current snapshot are present in the incremental file. The incremental file should contain all posts from the current snapshot plus any new posts added since the previous snapshot.

In [16]:
def check_snapshot_incremental_consistency(df_snapshot, df_incr, snapshot_name):
    """Check if all posts from snapshot are in incremental"""
    print(f"\n{'='*60}")
    print(f"Snapshot vs Incremental Consistency - {snapshot_name}")
    print(f"{'='*60}\n")
    
    if df_snapshot is None or df_incr is None:
        print("⚠️  Cannot perform check: one or both dataframes are None")
        return
    
    # Get post IDs
    snapshot_ids = set(df_snapshot['id'].dropna())
    incremental_ids = set(df_incr['id'].dropna())
    
    print(f"Posts in current snapshot: {len(snapshot_ids):,}")
    print(f"Posts in incremental: {len(incremental_ids):,}")
    
    # Check if all snapshot posts are in incremental
    missing_in_incremental = snapshot_ids - incremental_ids
    extra_in_incremental = incremental_ids - snapshot_ids
    common_ids = snapshot_ids & incremental_ids
    
    print(f"\nPosts in both files: {len(common_ids):,}")
    print(f"Posts only in snapshot: {len(missing_in_incremental):,}")
    print(f"Posts only in incremental: {len(extra_in_incremental):,}")
    
    # Calculate percentages
    if len(snapshot_ids) > 0:
        coverage_pct = (len(common_ids) / len(snapshot_ids)) * 100
        print(f"\nSnapshot coverage in incremental: {coverage_pct:.2f}%")
    
    # Verdict
    if len(missing_in_incremental) == 0:
        print(f"\n✓ All posts from snapshot are present in incremental")
    else:
        print(f"\n⚠️  {len(missing_in_incremental)} posts from snapshot are MISSING in incremental!")
        print(f"Sample missing IDs: {list(missing_in_incremental)[:10]}")
    
    if len(extra_in_incremental) > 0:
        print(f"\n✓ Incremental contains {len(extra_in_incremental):,} additional posts not in snapshot")
    
    return {
        'snapshot_count': len(snapshot_ids),
        'incremental_count': len(incremental_ids),
        'common_count': len(common_ids),
        'missing_in_incremental': len(missing_in_incremental),
        'extra_in_incremental': len(extra_in_incremental)
    }

# Check consistency for current snapshot
if df_current is not None and df_incremental is not None:
    consistency_results = check_snapshot_incremental_consistency(
        df_current, df_incremental, snapshot.name
    )


Snapshot vs Incremental Consistency - Jul 25

Posts in current snapshot: 18,926
Posts in incremental: 18,926

Posts in both files: 18,926
Posts only in snapshot: 0
Posts only in incremental: 0

Snapshot coverage in incremental: 100.00%

✓ All posts from snapshot are present in incremental


## Data Quality Check 6: Cross-Snapshot Consistency

Check consistency across all available snapshots to ensure incremental files contain all posts from their corresponding snapshots.

In [17]:
# Check all snapshots
print("Checking all snapshots...")
print("="*60)

all_results = []

for snap_folder in sorted(snapshot_folders):
    print(f"\n\nAnalyzing: {snap_folder.name}")
    print("-"*60)
    
    current_path = snap_folder / 'posts_current_snapshot.csv'
    incr_path = snap_folder / 'posts_incremental.csv'
    
    if not current_path.exists():
        print(f"⚠️  posts_current_snapshot.csv not found")
        continue
    
    if not incr_path.exists():
        print(f"⚠️  posts_incremental.csv not found")
        continue
    
    try:
        # Load only the id column for efficiency
        df_snap = pd.read_csv(current_path, usecols=['id'])
        df_incr = pd.read_csv(incr_path, usecols=['id'])
        
        # Get post IDs
        snapshot_ids = set(df_snap['id'].dropna())
        incremental_ids = set(df_incr['id'].dropna())
        
        # Calculate metrics
        missing = snapshot_ids - incremental_ids
        extra = incremental_ids - snapshot_ids
        common = snapshot_ids & incremental_ids
        
        coverage_pct = (len(common) / len(snapshot_ids) * 100) if len(snapshot_ids) > 0 else 0
        
        result = {
            'snapshot': snap_folder.name,
            'snapshot_posts': len(snapshot_ids),
            'incremental_posts': len(incremental_ids),
            'common_posts': len(common),
            'missing_in_incremental': len(missing),
            'extra_in_incremental': len(extra),
            'coverage_pct': coverage_pct
        }
        
        all_results.append(result)
        
        # Print summary
        print(f"Snapshot: {len(snapshot_ids):,} posts")
        print(f"Incremental: {len(incremental_ids):,} posts")
        print(f"Common: {len(common):,} posts")
        print(f"Coverage: {coverage_pct:.2f}%")
        
        if len(missing) > 0:
            print(f"⚠️  Missing in incremental: {len(missing):,} posts")
        else:
            print(f"✓ All snapshot posts in incremental")
            
    except Exception as e:
        print(f"Error processing snapshot: {e}")

# Create summary dataframe
if all_results:
    print("\n\n" + "="*80)
    print("SUMMARY - All Snapshots")
    print("="*80)
    results_df = pd.DataFrame(all_results)
    display(results_df) # ty: ignore[unresolved-reference]
    
    # Overall statistics
    total_snapshots = len(results_df)
    perfect_coverage = len(results_df[results_df['coverage_pct'] == 100.0])
    
    print(f"\n✓ Snapshots with 100% coverage: {perfect_coverage}/{total_snapshots}")
    if perfect_coverage < total_snapshots:
        print(f"⚠️  Snapshots with incomplete coverage: {total_snapshots - perfect_coverage}/{total_snapshots}")

Checking all snapshots...


Analyzing: 2016-2021
------------------------------------------------------------
Snapshot: 18,926 posts
Incremental: 18,926 posts
Common: 18,926 posts
Coverage: 100.00%
✓ All snapshot posts in incremental


Analyzing: 2022
------------------------------------------------------------
Snapshot: 13,108 posts
Incremental: 32,034 posts
Common: 13,108 posts
Coverage: 100.00%
✓ All snapshot posts in incremental


Analyzing: 2023
------------------------------------------------------------
Snapshot: 13,376 posts
Incremental: 45,410 posts
Common: 13,376 posts
Coverage: 100.00%
✓ All snapshot posts in incremental


Analyzing: 2024
------------------------------------------------------------
Snapshot: 13,400 posts
Incremental: 58,810 posts
Common: 13,400 posts
Coverage: 100.00%
✓ All snapshot posts in incremental


Analyzing: Jan-Jul 25
------------------------------------------------------------
Snapshot: 44,878 posts
Incremental: 103,688 posts
Common: 44,878 posts
C

,snapshot,snapshot_posts,incremental_posts,common_posts,missing_in_incremental,extra_in_incremental,coverage_pct
0,2016-2021,18926,18926,18926,0,0,100.0
1,2022,13108,32034,13108,0,18926,100.0
2,2023,13376,45410,13376,0,32034,100.0
3,2024,13400,58810,13400,0,45410,100.0
4,Jan-Jul 25,44878,103688,44878,0,58810,100.0
5,Jul 25,78857,182545,78857,0,103688,100.0



✓ Snapshots with 100% coverage: 6/6
